#Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Create Flag Parameter

In [0]:
# Check whether the Gold dimension table already exists
if spark.catalog.tableExists("store_catalog.gold.dim_crm_prd_info"):
    incremental_flag = "1"
else:
    incremental_flag = "0"

In [0]:
print(incremental_flag)

1


#Create Dimension crm_prd_info

###Fetch Relative Columns

In [0]:
# Load distinct product records from the Silver layer
df_scr = spark.sql('''
        SELECT DISTINCT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_prd_info`
    ''')


In [0]:
df_scr.limit(10).display()

product_id,product_number,product_name,product_cost,product_line,start_date,end_date,category_id,record_priority
559,CH-0234,Chain,9,n/a,2013-07-01,null,CO_CH,1
315,BK-R68R-58,Road-450 Red- 58,885,Road,2011-07-01,2007-12-28,BI_RB,1
374,BK-R89B-44,Road-250 Black- 44,1555,Road,2013-07-01,null,BI_RB,1
602,BB-8107,ML Bottom Bracket,45,n/a,2013-07-01,null,CO_BB,1
287,FR-R38B-52,LL Road Frame - Black- 52,205,Road,2013-07-01,null,CO_RF,1
456,TG-W091-S,Women's Tights- S,31,Other Sales,2012-07-01,2008-12-27,CL_TI,1
463,GL-H102-S,Half-Finger Gloves- S,9,Other Sales,2013-07-01,null,CL_GL,1
503,FR-T67U-54,LL Touring Frame - Blue- 54,200,Touring,2013-07-01,null,CO_TF,1
511,FR-M63S-40,ML Mountain Frame-W - Silver- 40,199,Mountain,2013-07-01,null,CO_MF,1
373,BK-R89B-44,Road-250 Black- 44,1321,Road,2012-07-01,2008-12-27,BI_RB,1


###dim_crm_prd_info - Initial and Incremental

In [0]:
# Read existing Gold records or create an empty DataFrame for the initial load
if spark.catalog.tableExists("store_catalog.gold.dim_crm_prd_info") == True:
    df_sink = spark.sql('''
            SELECT
                crm_prd_info_surrogate_key,
                product_id,
                product_number,
                product_name,
                product_cost,
                product_line,
                start_date,
                end_date,
                category_id
            FROM
                store_catalog.gold.dim_crm_prd_info
        ''')

else:
    df_sink = spark.sql('''
            SELECT
                1 as crm_prd_info_surrogate_key,
                product_id,
                product_number,
                product_name,
                product_cost,
                product_line,
                start_date,
                end_date,
                category_id
            FROM
                parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_prd_info`
            WHERE
                1 = 0 
        ''')


###Filtering new records and old records

In [0]:
# Match Silver records with existing Gold records by product ID
df_filter = df_scr.join(df_sink,df_scr["product_id"] == df_sink["product_id"], "left")\
        .select(df_scr["product_id"],df_scr["product_number"],df_scr["product_name"],
            df_scr["product_cost"],df_scr["product_line"],df_scr["start_date"],
            df_scr["end_date"],df_scr["category_id"],
            df_sink["crm_prd_info_surrogate_key"]
    )

In [0]:
df_filter.groupBy("product_id") \
    .count() \
    .filter(col("count") > 1) \
    .limit(10)\
    .show(truncate=False)

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



**df_filter_old**

In [0]:
# Separate existing product records
df_filter_old = df_filter.filter(col("crm_prd_info_surrogate_key").isNotNull())

**df_filter_new**

In [0]:
# Select new products that require surrogate keys
df_filter_new = df_filter.filter(col("crm_prd_info_surrogate_key").isNull())\
    .select(df_scr["product_id"],df_scr["product_number"],df_scr["product_name"],
            df_scr["product_cost"],df_scr["product_line"],df_scr["start_date"],
            df_scr["end_date"],df_scr["category_id"]
    )

In [0]:
df_filter_new.limit(10).display()

product_id,product_number,product_name,product_cost,product_line,start_date,end_date,category_id


###Create Surrogate Key

**Fetch the max Surrogate Key from existing table**

In [0]:
# Retrieve the next available surrogate key
if (incremental_flag == "0"):
    max_value = 1
else:
    max_value_df = spark.sql('''SELECT MAX(crm_prd_info_surrogate_key) FROM store_catalog.gold.dim_crm_prd_info''')
    max_value = max_value_df.collect()[0][0]+1

**Create Surrogate Key column and Add the max surrogate key**

In [0]:
# Assign unique surrogate keys to new product records
df_filter_new = df_filter_new.withColumn("crm_prd_info_surrogate_key",max_value + monotonically_increasing_id())
df_filter_new.limit(10).display()

product_id,product_number,product_name,product_cost,product_line,start_date,end_date,category_id,crm_prd_info_surrogate_key


###Create Final DF - df_filter_old + df_filter_new

In [0]:
# Combine new products with existing product records
df_final = df_filter_new.union(df_filter_old)

In [0]:
df_final.limit(10).display()

product_id,product_number,product_name,product_cost,product_line,start_date,end_date,category_id,crm_prd_info_surrogate_key
559,CH-0234,Chain,9,n/a,2013-07-01,null,CO_CH,1
315,BK-R68R-58,Road-450 Red- 58,885,Road,2011-07-01,2007-12-28,BI_RB,2
374,BK-R89B-44,Road-250 Black- 44,1555,Road,2013-07-01,null,BI_RB,3
602,BB-8107,ML Bottom Bracket,45,n/a,2013-07-01,null,CO_BB,4
287,FR-R38B-52,LL Road Frame - Black- 52,205,Road,2013-07-01,null,CO_RF,5
456,TG-W091-S,Women's Tights- S,31,Other Sales,2012-07-01,2008-12-27,CL_TI,6
463,GL-H102-S,Half-Finger Gloves- S,9,Other Sales,2013-07-01,null,CL_GL,7
503,FR-T67U-54,LL Touring Frame - Blue- 54,200,Touring,2013-07-01,null,CO_TF,8
511,FR-M63S-40,ML Mountain Frame-W - Silver- 40,199,Mountain,2013-07-01,null,CO_MF,9
373,BK-R89B-44,Road-250 Black- 44,1321,Road,2012-07-01,2008-12-27,BI_RB,10


# SCD TYPE - 1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("store_catalog.gold.dim_crm_prd_info"):
    # Incremental Run
    # Merge product records into the Gold dimension table
    delta_tbl = DeltaTable.forPath(spark,"abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_crm_prd_info")

    delta_tbl.alias("trg").merge(df_final.alias("src"),"trg.crm_prd_info_surrogate_key = src.crm_prd_info_surrogate_key")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()

else:
    # Initial Run
    # Create the Gold dimension table during the initial load
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_crm_prd_info")\
        .saveAsTable("store_catalog.gold.dim_crm_prd_info")


In [0]:
%sql
SELECT * FROM store_catalog.gold.dim_crm_prd_info ORDER BY crm_prd_info_surrogate_key DESC LIMIT 10

product_id,product_number,product_name,product_cost,product_line,start_date,end_date,category_id,crm_prd_info_surrogate_key
526,FR-M21S-48,LL Mountain Frame - Silver- 48,145,Mountain,2013-07-01,null,CO_MF,398
549,FR-M63S-38,ML Mountain Frame-W - Silver- 38,199,Mountain,2013-07-01,null,CO_MF,397
504,FR-T67U-58,LL Touring Frame - Blue- 58,200,Touring,2013-07-01,null,CO_TF,396
370,BK-R89R-52,Road-250 Red- 52,1519,Road,2012-07-01,2008-12-27,BI_RB,395
224,CA-1098,AWC Logo Cap,5,Other Sales,2012-07-01,2008-12-27,CL_CA,394
464,GL-H102-M,Half-Finger Gloves- M,10,Other Sales,2012-07-01,2008-12-27,CL_GL,393
275,FR-R72R-48,ML Road Frame - Red- 48,352,Road,2011-07-01,2007-12-28,CO_RF,392
367,BK-M47B-48,Mountain-300 Black- 48,598,Mountain,2012-07-01,2008-12-27,BI_MB,391
277,FR-R72R-58,ML Road Frame - Red- 58,352,Road,2011-07-01,2007-12-28,CO_RF,390
257,FR-R38B-60,LL Road Frame - Black- 60,170,Road,2012-07-01,2008-12-27,CO_RF,389


In [0]:
%sql
SELECT COUNT(*) FROM store_catalog.gold.dim_crm_prd_info 

count(1)
398
